# Preparation of Experimantal Data

## Testing Google Spreadsheet Connection

In [7]:
import gspread
import pandas as pd

# 1. Trigger the browser login using your downloaded JSON file
# It will save a second file ('authorized_user.json') so you don't have to log in every single time.
gc = gspread.oauth(
    credentials_filename='credentials.json',
    authorized_user_filename='authorized_user.json'
)

# 2. Open the Google Sheet using its URL
# Replace this with the actual URL of your HPLC Google Sheet
SHEET_URL = "https://docs.google.com/spreadsheets/d/13ihUtinfoi9AmLZtnqF6BX7L-dqAk552I-KZNu2HtFc/edit#gid=0"
workbook = gc.open_by_url(SHEET_URL)

# 3. Load the Standard tab into Pandas
worksheet_name = "data" 
worksheet = workbook.worksheet(worksheet_name)

# 4. Convert the data into a Pandas DataFrame
data = worksheet.get_all_records()
df_std = pd.DataFrame(data)

# 5. Display the first 5 rows to verify!
display(df_std.head())

,Perbandingan,Waktu Sampling,Peak Name,Zone,Peak Identification,RT,Area,% Area,Conc
0,1.0,t0,desB30 II,0,desb30,4.755,3178704,99.63,1.44874
1,1.0,t2,desB30 II,0,desb30,4.771,992159,32.94,0.45486
2,1.0,t2,Peptide Impurity I,1,A1,5.807,63584,2.11,0.03278
3,1.0,t2,Detemir,1,detemir,5.938,1751603,58.15,0.80006
4,1.0,t2,Peptide Impurity II,2,A1_B29,6.481,171663,5.70,0.08191


## Experimental Data Prep for Modeling

In [11]:
import pandas as pd
import numpy as np

# 1. Define the sheets we want to process and their corresponding "Perbandingan" values
sheet_mapping = {
    "1:1": 1.0, 
    "1:1.5": 1.5, 
    "1:2": 2.0
}

# List to hold the processed dataframes before we combine them
processed_dfs = []

for sheet_name, perbandingan_val in sheet_mapping.items():
    
    # 2. Open the worksheet
    # NOTE: If your tab names have the full prefix, change this to f"Data HPLC NHS-DesB30 - {sheet_name}"
    ws = workbook.worksheet(sheet_name) 
    
    # Load into Pandas
    data = ws.get_all_records()
    df = pd.DataFrame(data)
    
    # 3. Select only the 3 specific columns
    df = df[['Kode Sampel', 'Peak Identification', 'Conc']]
    
    # 4. Filter out empty "Peak Identification" rows
    # We replace any purely blank spaces with NaN, then drop the NaNs
    df['Peak Identification'] = df['Peak Identification'].replace(r'^\s*$', np.nan, regex=True)
    df = df.dropna(subset=['Peak Identification'])
    
    # 5. Insert the "Perbandingan" column at position 0 (before "Kode Sampel")
    df.insert(0, 'Perbandingan', perbandingan_val)
    
    # 6. Ensure the data types are correct
    df['Perbandingan'] = pd.to_numeric(df['Perbandingan'])
    
    # For 'Conc', sometimes Google Sheets passes numbers as strings with commas. 
    # This ensures it forces them into strict numeric formats safely.
    df['Conc'] = pd.to_numeric(df['Conc'].astype(str).str.replace(',', ''), errors='coerce') 
    
    df['Kode Sampel'] = df['Kode Sampel'].astype(str)
    df['Peak Identification'] = df['Peak Identification'].astype(str)
    
    # Add the cleaned dataframe to our list
    processed_dfs.append(df)

# 7. Combine everything into a single DataFrame
exp_data = pd.concat(processed_dfs, ignore_index=True)

# Display the result and the data types to verify!
display(exp_data.head())
print("\n--- Data Types ---")
print(exp_data.dtypes)

,Perbandingan,Kode Sampel,Peak Identification,Conc
0,1.0,t0,desb30,1.44520
1,1.0,t2,desb30,0.45109
2,1.0,t2,A1,0.02891
3,1.0,t2,detemir,0.79637
4,1.0,t2,A1_B29,0.07805



--- Data Types ---
Perbandingan           float64
Kode Sampel             object
Peak Identification     object
Conc                   float64
dtype: object


In [12]:
import pandas as pd
import numpy as np

# 1. Clean Time column (convert "t2" -> 2.0)
exp_data['Time'] = exp_data['Kode Sampel'].str.replace('t', '', regex=False).astype('float64')

# 2. Pivot the data WITHOUT filling missing values. 
# This finds every unique peak that ever appeared in the dataset and creates a column for it.
# Intermediate missing values remain as NaN.
exp_data_grid = exp_data.pivot_table(
    index=['Perbandingan', 'Time'], 
    columns='Peak Identification', 
    values='Conc'
).reset_index()

# 3. Melt it back into a long format
final_kinetic_data = exp_data_grid.melt(
    id_vars=['Perbandingan', 'Time'],
    value_name='Conc'
)

# 4. Tag Substrate vs. Product (Addressing Point 4)
# np.where works like an Excel IF statement: if it's desb30, label it Substrate, else Product.
final_kinetic_data['Type'] = np.where(
    final_kinetic_data['Peak Identification'].str.lower() == 'desb30', 
    'Substrate', 
    'Product'
)

# 5. The Initial Value Fix (Addressing Points 1 & 2)
# Find rows where Time is 0 AND the Type is Product, then explicitly set their Conc to 0.0
mask_t0_products = (final_kinetic_data['Time'] == 0.0) & (final_kinetic_data['Type'] == 'Product')
final_kinetic_data.loc[mask_t0_products, 'Conc'] = 0.0

# 6. Clean up intermediate missing values (Addressing Point 3)
# We drop rows that are STILL NaN (e.g., the t=40 missing data point).
final_kinetic_data = final_kinetic_data.dropna(subset=['Conc'])

# 7. Reorder columns and sort for a clean view
final_kinetic_data = final_kinetic_data[['Perbandingan', 'Time', 'Type', 'Peak Identification', 'Conc']]
final_kinetic_data = final_kinetic_data.sort_values(
    by=['Perbandingan', 'Peak Identification', 'Time']
).reset_index(drop=True)

# Display the result
display(final_kinetic_data.head(15))

,Perbandingan,Time,Type,Peak Identification,Conc
0,1.0,0.0,Product,A1,0.00000
1,1.0,2.0,Product,A1,0.02891
2,1.0,4.0,Product,A1,0.02600
3,1.0,6.0,Product,A1,0.02446
4,1.0,8.0,Product,A1,0.02529
5,1.0,10.0,Product,A1,0.02253
6,1.0,13.0,Product,A1,0.02084
7,1.0,16.0,Product,A1,0.02037
8,1.0,19.0,Product,A1,0.02093
9,1.0,22.0,Product,A1,0.02290


In [13]:
display(final_kinetic_data)

,Perbandingan,Time,Type,Peak Identification,Conc
0,1.0,0.0,Product,A1,0.00000
1,1.0,2.0,Product,A1,0.02891
2,1.0,4.0,Product,A1,0.02600
3,1.0,6.0,Product,A1,0.02446
4,1.0,8.0,Product,A1,0.02529
...,...,...,...,...,...
384,2.0,60.0,Product,detemir,0.36382
385,2.0,70.0,Product,detemir,0.34331
386,2.0,80.0,Product,detemir,0.36898
387,2.0,100.0,Product,detemir,0.33011


In [14]:
# Save the clean data to a CSV file
file_name = 'kinetic_data_processed.csv'
final_kinetic_data.to_csv(file_name, index=False)

print(f"Dataset successfully saved as {file_name}")

Dataset successfully saved as kinetic_data_processed.csv
